# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset Croissant URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print the dataset metadata
meta_dict = dataset.metadata.to_json()
print(f"{meta_dict['name']}: {meta_dict['description']}")
print(f"License: {meta_dict['license']}")
print(f"Croissant version: {meta_dict['conformsTo']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
from typing import List

# List all available record sets and their @id
print("Available Record Sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']}")

example_record_set_id = dataset.metadata.record_sets[0]['@id']

# List fields and columns for each record set
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    print("Fields:")
    for field in record_set['fields']:
        print(f"  - {field['@id']}: {field['name']}")
        # If available, print columns
        if 'columns' in field:
            for col in field['columns']:
                print(f"      - column @id: {col['@id']}, name: {col['name']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set(s) by @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records_gen = dataset.records(record_set=rs_id)
    # Convert generator to list for safety
    records = list(records_gen)
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records with columns: ", df.columns.tolist())

# Example: show first 5 rows for the first record set
main_rs = record_set_ids[0]
columns = dataframes[main_rs].columns.tolist()
print(f"\nSample records from record set {main_rs}:")
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field from the main record set
main_df = dataframes[main_rs]

# Let's inspect all columns to pick a numeric field and group field
print("Available columns:", main_df.columns.tolist())

# We'll try using 'MW' (Molecular Weight) and group by 'Description' if present
numeric_field = None
possible_numeric_fields = [
    col for col in main_df.columns if any(k in col.lower() for k in ['mw', 'weight', 'count', 'abundance', 'coverage', 'number', 'quant'])
]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    print("No clear numeric fields found, please inspect manually.")

# Sample filtering and normalization
if numeric_field:
    try:
        # Convert to numeric, coerce errors
        main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
        threshold = main_df[numeric_field].mean()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value)")
        print(filtered_df[[numeric_field]].head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} example:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        possible_group_fields = [col for col in main_df.columns if ('desc' in col.lower()) or ('class' in col.lower()) or ('type' in col.lower())]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nGrouped by {group_field}:\n", grouped_df.head())
        else:
            print("No group field found for grouping.")
    except Exception as e:
        print(f"Error during EDA: {e}")
else:
    print("No numeric field to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field if available
if numeric_field and not main_df[numeric_field].isnull().all():
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² mass spectrometry dataset includes <b>protein abundance</b>, <b>modification</b>, and <b>sequence</b> data, with rich metadata accessible through the Croissant specification.
- Using `mlcroissant`, we listed all available record sets, fields, and analyzed numeric fields such as molecular weight (if present), demonstrating basic EDA and plotting.
- For deeper insights: explore additional fields using their `@id`, reference the schema for advanced querying, and customize filtering/grouping based on research needs.